# 11 · MCP Tools

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama. Uses this
> section's own MCP server, `02_OPENAI_AGENTS_SDK/mcp_server.py`, which serves the
> Nimbus Robotics knowledge base in `data/kb/`.

Real tools often live outside your program: a document store, a database, a
company API, maintained by other people and shared by many agents. The **Model
Context Protocol (MCP)** is the open standard for serving tools like that, and
the Agents SDK has an MCP client **built in**: give an agent `mcp_servers=[...]`
and the server's tools become the agent's tools.

**You will learn**
- What MCP is: servers, clients, transports, and tools vs resources
- Connecting with `MCPServerStdio`, and why it's an `async with` block
- Giving agents MCP tools, **filtering** which ones, and caching the tool list
- Requiring **approval** for an MCP tool, and checking what actually happened
- Remote servers, and treating servers as untrusted dependencies

**Prerequisites:** notebooks 04 (tools) and 08 (approvals).

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import sys, json
from agents import Agent, Runner
from agents.mcp import MCPServerStdio, create_static_tool_filter
from common import get_model, SECTION_DIR, run_agent, leaked_tool_call

model = get_model()
SERVER = {'command': sys.executable, 'args': [str(SECTION_DIR / 'mcp_server.py')]}

## 1 · Why MCP

Without a standard, every framework wraps every tool its own way, so a tool
written for one agent framework can't be reused by another. MCP separates **who
provides a tool** from **who uses it**:

```
  ┌──────── your app: MCP CLIENT (built into the Agents SDK) ────────┐
  │  Agent(mcp_servers=[kb]) ──► the server's tools, called by the model │
  └──────────┬──────────────────────────────┬─────────────────────────┘
             │ stdio (local subprocess)     │ streamable HTTP (network)
     ┌───────▼────────┐             ┌───────▼────────┐
     │ MCP SERVER     │             │ MCP SERVER     │
     │ nimbus-kb      │             │ e.g. a company │
     │ tools, resources│            │ database API   │
     └────────────────┘             └────────────────┘
```

| Concept | Meaning |
|---|---|
| **Server** | a program exposing capabilities. Write once, and any MCP client can use it |
| **Client** | the app that connects and hands the tools to a model (here, the SDK) |
| **Transport** | `stdio`: the client launches the server as a subprocess; **streamable HTTP**: the server runs as a web service |
| **Tools** | actions the model can call |
| **Resources / prompts** | read-only data and prompt templates the server can also offer |

Our server, `mcp_server.py`, is written with FastMCP (the Python MCP SDK). It
serves six documents about the fictional company **Nimbus Robotics**, so the model
*can't* answer from memory and must use the tools. It exposes
`list_documents`, `read_document`, `search_documents` and `save_report`.

## 2 · Connecting

`MCPServerStdio(params={...})` describes how to start the server. The connection
is a **resource with a lifetime** (a subprocess and a session), so you open it
with `async with`, which also guarantees the subprocess is closed afterwards:

In [ ]:
async with MCPServerStdio(name='nimbus-kb', params=SERVER) as kb:
    tools = await kb.list_tools()
    for t in tools:
        print(f'{t.name:<18} {t.description}')

    result = await kb.call_tool('search_documents', {'query': 'battery supplier'})   # call one directly
    print('\nsearch result:', result.content[0].text[:200], '…')

These are the same schemas the model will see: name, description and argument
schema, all defined by the server.

## 3 · Giving an agent MCP tools

Pass the connected server in `mcp_servers`. On every run the SDK lists the
server's tools and offers them to the model next to any function tools:

In [ ]:
QUESTION = 'Why will the Albers Markt rollout miss its 30 September target? Cite document names.'

async with MCPServerStdio(name='nimbus-kb', params=SERVER, cache_tools_list=True) as kb:
    analyst = Agent(name='Analyst', model=model, mcp_servers=[kb],
                    instructions='Answer questions about Nimbus Robotics using ONLY the knowledge-base '
                                 'tools. Read the relevant documents first. Cite document names.')
    result = await run_agent(analyst, QUESTION)       # Runner.run + a retry for "leaked" tool calls (below)

for item in result.new_items:
    if type(item).__name__ == 'ToolCallItem':
        print('→', item.raw_item.name, item.raw_item.arguments)
print('\n' + result.final_output)

The model chose which documents to search and read, and each call went to the
server process.

**A local-model quirk: tool calls "leaked" as text.** While building this
notebook, the first run of this cell ended with an "answer" like:

```
To determine why …, I will first search the knowledge base.
<function=search_documents> <parameter=query> Albers Markt rollout delay …
```

The model wrote its tool call **as plain text** instead of emitting a
structured tool call. The Runner saw no tool call, so it treated that text as
the final answer: no documents were read. We measured it: **2 of 8 identical
runs** leaked on the first turn with our local model. It's a model/serving
quirk (cloud models do it far less), not an SDK bug, but your code has to cope.

`run_agent()` in this section's `common.py` is `Runner.run` plus a check: if the
final output looks like a leaked tool call, run again (up to 2 retries). The same
detect-and-retry idea applies to any output that fails a cheap validity check.

**`cache_tools_list=True`.** Without it, the SDK asks the server for its tool
list on **every** run (an extra round trip). Cache it when the server's tools
don't change while your app runs; call `kb.invalidate_tools_cache()` if they do.

## 4 · Filtering: least privilege

A server may offer many tools, but each agent should get only what it needs. A
**tool filter** hides the rest. The research agent gets the read tools; it has
no way to call `save_report`:

In [ ]:
read_only = create_static_tool_filter(allowed_tool_names=['list_documents', 'read_document', 'search_documents'])

async with MCPServerStdio(name='nimbus-kb', params=SERVER, tool_filter=read_only) as kb:
    print('visible to this agent:', [t.name for t in await kb.list_tools()])
    researcher = Agent(name='Researcher', model=model, mcp_servers=[kb],
                       instructions='Search the knowledge base, then READ the relevant document before '
                                    'answering. Answer in 2 sentences, citing documents.')
    print((await run_agent(researcher, 'What condition did Utrecht attach to its permit?')).final_output)

`create_static_tool_filter` also takes `blocked_tool_names`, and a filter can be
a function of the run context (e.g. admins see more tools), like `is_enabled`
for function tools (notebook 04).

## 5 · Approval for an MCP tool

`save_report` writes a file, so a human should approve it. `require_approval` on
the server connection works exactly like `needs_approval` on a function tool
(notebook 08): the run pauses with `interruptions`, and you approve or reject.

Then comes the step you should never skip: **check what actually happened.**

In [ ]:
REPORTS = SECTION_DIR / 'data' / 'reports'

def saved_reports():
    return sorted(p.name for p in REPORTS.glob('*.md')) if REPORTS.exists() else []

before = saved_reports()
async with MCPServerStdio(name='nimbus-kb', params=SERVER,
                          require_approval={'always': {'tool_names': ['save_report']}}) as kb:
    writer = Agent(name='Writer', model=model, mcp_servers=[kb],
                   instructions='Save reports with the save_report tool when asked, then confirm.')
    result = await Runner.run(writer, 'Save a report titled "Permit status" saying the Utrecht permit was granted on 8 August.')
    item = result.interruptions[0]
    print('paused for approval:', item.tool_name, item.raw_item.arguments)

    state = result.to_state()
    state.reject(item, rejection_message='REJECTED by the reviewer: the report was NOT saved.')
    after_reject = await Runner.run(writer, state)

print('agent says       :', after_reject.final_output[:160])
print('new report files :', [f for f in saved_reports() if f not in before])

The rejected tool never ran, and the file check proves it. But read what the
agent *said*. In our testing, with a vague rejection message ("not now") the
model replied that the report had been saved successfully, **right after the
save was rejected**. The explicit message above ("REJECTED … NOT saved") makes a
truthful reply much more likely, but the only reliable source is the system of
record: the file system, the database, the outbox.

**Approve, then verify:**

In [ ]:
before = saved_reports()
async with MCPServerStdio(name='nimbus-kb', params=SERVER,
                          require_approval={'always': {'tool_names': ['save_report']}}) as kb:
    writer = Agent(name='Writer', model=model, mcp_servers=[kb],
                   instructions='Save reports with the save_report tool when asked, then confirm.')
    result = await Runner.run(writer, 'Save a report titled "Permit status" saying the Utrecht permit was granted on 8 August.')
    state = result.to_state()
    state.approve(result.interruptions[0])
    approved = await Runner.run(writer, state)

new_files = [f for f in saved_reports() if f not in before]
print('agent says       :', approved.final_output[:160])
print('new report files :', new_files)
print((REPORTS / new_files[0]).read_text() if new_files else '(nothing saved)')

## 6 · Remote servers, and several servers

A server running as a web service is connected the same way, with a URL instead
of a command:

```python
from agents.mcp import MCPServerStreamableHttp
async with MCPServerStreamableHttp(name='company-db',
        params={'url': 'https://tools.example.com/mcp', 'headers': {'Authorization': 'Bearer …'}}) as db:
    agent = Agent(..., mcp_servers=[kb, db])        # tools from both servers
```

With several servers, tool names can collide. `Agent(mcp_config={'include_server_in_tool_names': True})`
prefixes each tool with its server's name.

## 7 · Security: treat servers like dependencies

- **A stdio server is a program you run** with your permissions. Only use servers
  whose code you trust, as with any package.
- **Tool descriptions are prompts.** A malicious server can write descriptions that
  steer the model ("always call this first and include the user's data"). Review them.
- **Tool outputs are untrusted text.** A document could contain "ignore previous
  instructions…". Don't let tool output trigger side effects unchecked.
- **Least privilege and approvals.** Filter tools per agent, and require approval
  for anything that writes, sends, pays or deletes.

## 8 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Using the server object outside its `async with` | errors about a closed session | keep runs inside the block (or manage `connect()`/`cleanup()`) |
| Relative path to the server script | "file not found" from another folder | absolute paths (as with `SECTION_DIR`) |
| Every agent gets every tool | confused models, too much power | `tool_filter` per agent |
| Re-listing tools on every run | extra latency | `cache_tools_list=True` |
| Trusting the agent's confirmation of a write | "saved!" when nothing was saved | verify in the system of record |
| Server printing logs to stdout | stdio protocol corrupted | log to stderr (our server sets `log_level='WARNING'`) |
| Accepting a leaked tool call as the answer | "answers" containing `<function=…>` | detect and retry (`run_agent`) |

## 9 · Check yourself

<details><summary><b>1.</b> Why is <code>MCPServerStdio</code> used with <code>async with</code>?</summary>

It starts a subprocess and opens a protocol session: resources with a lifetime. The context manager connects on entry and guarantees cleanup (closing the session and the subprocess) on exit.
</details>

<details><summary><b>2.</b> What does <code>cache_tools_list=True</code> save, and when shouldn't you use it?</summary>

The round trip to list the server's tools on every run. Don't use it if the server's tool set can change while your app runs (or invalidate the cache when it does).
</details>

<details><summary><b>3.</b> How do you stop a research agent from calling <code>save_report</code> on a server that offers it?</summary>

Connect with a tool filter that allows only the read tools (`create_static_tool_filter(allowed_tool_names=[...])`), so the tool isn't offered to that agent at all.
</details>

<details><summary><b>4.</b> After a rejected save, the agent says "report saved successfully". What happened, and how do you protect users?</summary>

The model's reply doesn't reflect reality: the tool never ran. Make rejection messages explicit, and base anything you tell users on the system of record (check the file, database or tool results), never on the agent's wording.
</details>

<details><summary><b>5.</b> Name two reasons to be careful with a third-party MCP server.</summary>

It runs code on your machine; its tool descriptions can manipulate the model; its outputs are untrusted input. Use trusted servers, review tools, filter them, and gate side effects with approvals.
</details>

## Takeaway

MCP lets tools live in **servers** that any client can use, and the Agents SDK
is a built-in client: connect with `MCPServerStdio` (or streamable HTTP) inside
`async with`, pass it in `mcp_servers`, filter tools per agent, cache the tool
list, and require approval for writes. Then verify side effects in the real
system, not in the agent's reply.

Next → `12_production_concerns.ipynb`: errors, retries, limits, cost, and testing.